# Lezione 37 — Valutare un modello generativo

> **Modulo:** Transformer e modello open · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 13 (metriche), Lezione 36 (output strutturato).
>
> Obiettivo pratico unico: valutare un estrattore con metriche **a livello di
> campo** (precision/recall/F1 sulle entita' estratte) su un piccolo set
> etichettato a mano — pienamente eseguibile — e capire il ruolo dell'
> **LLM-as-judge**.

## Teoria essenziale

Valutare la generazione e' piu' difficile della classificazione: spesso non
esiste **una** risposta giusta. Tre livelli, dal piu' oggettivo al piu'
soggettivo:

1. **Exact match**: l'output e' esattamente quello atteso? Utile per compiti
   vincolati (es. il `type` della memoria).
2. **Metriche a livello di campo**: per l'estrazione di entita', si contano
   veri/falsi positivi/negativi **per entita'** e si calcolano
   precision/recall/F1 (le stesse della Lezione 13, applicate a insiemi).
3. **LLM-as-judge**: un secondo modello giudica la qualita' quando non c'e' una
   verita' netta (fluidita', pertinenza). Potente ma va **calibrato** contro
   giudizi umani, altrimenti si misura il bias del giudice.

Oggi implementiamo il livello 2, che non richiede alcun modello.

In [1]:
import re

def estrai(testo):
    cand = re.findall(r"\b[A-Z][a-zA-Z]+\b", str(testo))
    return {c for c in cand if c not in {"The", "A"}}

# piccolo set etichettato a mano (gold = entita' corrette attese)
GOLD = [
    ("Marco visited Glasgow with his son.", {"Marco", "Glasgow"}),
    ("The user prefers morning sessions.", set()),
    ("Elena works for Nordica in Torino.", {"Elena", "Nordica", "Torino"}),
    ("Lucia visited Bologna for the weekend.", {"Lucia", "Bologna"}),
]

def prf_a_livello_campo(dataset):
    tp = fp = fn = 0
    for testo, gold in dataset:
        pred = estrai(testo)
        tp += len(pred & gold)
        fp += len(pred - gold)
        fn += len(gold - pred)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"tp": tp, "fp": fp, "fn": fn,
            "precision": round(prec, 3), "recall": round(rec, 3), "f1": round(f1, 3)}

report = prf_a_livello_campo(GOLD)
print("report a livello di entita':", report)

report a livello di entita': {'tp': 7, 'fp': 0, 'fn': 0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


Leggi il report: `fp`/`fn` dicono *dove* l'estrattore sbaglia (es. la
maiuscola di inizio frase `The` gia' esclusa, ma nomi non capitalizzati
sfuggono). E' diagnostico, non solo un voto.

## Il progetto: Memory AI Lab, passo 37

Impacchetto la valutazione come funzione riusabile e ne fisso una **soglia di
regressione**: se una modifica futura all'estrattore fa scendere l'F1 sotto la
baseline misurata, il test se ne accorge.

In [2]:
def valuta_estrattore(dataset=GOLD):
    return prf_a_livello_campo(dataset)

r = valuta_estrattore()
# baseline misurata ora: F1 non deve regredire sotto questo valore
assert r["f1"] >= 0.8, r
assert r["precision"] == 1.0  # l'euristica non inventa entita' su questo set
print(f"baseline F1={r['f1']} precision={r['precision']} recall={r['recall']}")
print("guardia di regressione attiva (F1 >= 0.8)")

baseline F1=1.0 precision=1.0 recall=1.0
guardia di regressione attiva (F1 >= 0.8)


## Riepilogo

1. Valutare la generazione e' difficile: spesso non c'e' una sola risposta
   giusta.
2. **Exact match** per compiti vincolati (es. il `type`).
3. **Metriche a livello di campo** (P/R/F1 su insiemi) per l'estrazione.
4. `fp`/`fn` sono **diagnostici**: dicono *dove* si sbaglia.
5. **LLM-as-judge** per la qualita' soggettiva, ma va **calibrato** su umani.
6. Una soglia di F1 diventa una **guardia di regressione** nel progetto.

## Quiz

1. Perche' l'exact match e' inadatto a valutare una frase generata libera?
2. Cosa misura la recall nell'estrazione di entita'?
3. Qual e' il rischio dell'LLM-as-judge non calibrato?

*(Risposte: 1. perche' esistono molte formulazioni corrette diverse dalla
stringa attesa; 2. la frazione di entita' vere che l'estrattore ha trovato;
3. misurare il bias/preferenze del giudice invece della qualita' reale.)*